# Binary ASD vs TD — XLS-R (wav2vec2-xls-r-300m) deep-learning pipeline

**Two changes from the WavLM-Base-Plus notebook this is derived from:**
1. The encoder is **`facebook/wav2vec2-xls-r-300m`** instead of `microsoft/wavlm-base-plus`.
2. **Nothing is frozen.** Every transformer layer, the CNN feature extractor, and the head all train.

Everything else — the CHAT-timestamp cropping, the participant-level 5-fold CV, the leakage
assertions, the augmentation recipe, the occlusion explainability, the binary screening metrics — is
unchanged from that notebook.

# Based on "Evaluating Voice Biomarkers and Deep Learning for Neurodevelopmental Disorder Screening in Real-World Conditions"

Rakotomanana & Rouhafzay (2025), *Eng. Proc.* 2025, 118, 46. https://doi.org/10.3390/ECSA-12-26523

The base paper classifies **three** groups (ASD / ADHD / TD) with WavLM-Base-Plus. This notebook
narrows that to **two** (ASD / TD) on the SK sub-corpus of the Asymmetries Project (TalkBank/CHILDES)
and swaps the encoder. Both are deliberate deviations — see the tables below.

### Why XLS-R for Bangla speech

| | WavLM-Base-Plus | **XLS-R 300m (this notebook)** |
|---|---|---|
| Pretraining data | 94k h, **English only** (Libri-Light + GigaSpeech + VoxPopuli-en) | 436k h, **128 languages** (CommonVoice, VoxPopuli, MLS, BABEL, VoxLingua107) |
| Bengali/Bangla seen? | **no** | **yes** — Bangla appears in the CommonVoice and VoxLingua107 portions |
| Parameters | ~94 M | **~315 M** |
| Transformer layers | 12 | **24** |
| Hidden size | 768 | **1024** |
| Input normalisation | `do_normalize=False` (raw waveform) | **`do_normalize=True` — required, see below** |

If the speech in your corpus is Bangla, WavLM's phonetic representations were never fit to it. XLS-R's
were. That is the reason for the swap.

### Three things that changed *because* of the swap, not by choice

**1. Input normalisation is now mandatory.** XLS-R is a `feat_extract_norm="layer"` model, which means
its CNN front-end expects the raw waveform to be **zero-mean, unit-variance normalised per utterance**.
WavLM-Base-Plus is `feat_extract_norm="group"` with `do_normalize=False`, so the old notebook could
hand `librosa.load()` output straight to the model — and it did; the `Wav2Vec2FeatureExtractor` it
constructed was never actually called. Feeding un-normalised audio to XLS-R does not raise; it just
trains badly. Section 5 now normalises explicitly, in the dataset **and** in `predict_proba` so the
occlusion analysis sees the same distribution the model was trained on.

**2. Pooling is a learned weighted sum over all 24 layers, not a fixed mid-layer.** This follows from
"freeze nothing". With fixed-layer pooling at `hidden_states[k]`, the layers **above** `k` receive
exactly zero gradient no matter what `requires_grad` says — they are computed on every forward pass
and contribute nothing. Unfreezing them would be cosmetic. A learned softmax weight per layer (the
standard SUPERB / s3prl recipe for XLS-R) makes every layer both used and trained, which is what
"freeze nothing" has to mean here. `POOL_MODE` in the config cell still offers `"fixed_layer"` and
`"last_layer"` if you want to match the old structure exactly.

**3. The training recipe had to change to survive full fine-tuning.** A 315 M-parameter full
fine-tune on a T4 is a different problem from training two layers and a head:

| | WavLM notebook | This notebook | why |
|---|---|---|---|
| Trainable params | ~14 M (2 layers + head) | **~315 M (everything)** | as requested |
| Batch size | 8 | 4 x 2 accumulation = **8 effective** | 16 GB VRAM |
| Mixed precision | off | **fp16 AMP** | ~2x, and it is what makes 315 M fit |
| Gradient checkpointing | off | **on** | trades ~30% speed for ~40% memory |
| LR warmup | none | **10% linear warmup** | wav2vec2-family full FT diverges without it |
| Gradient clipping | none | **max-norm 1.0** | same reason |
| Epochs | 25 | **10, with early stopping** | full FT converges in far fewer, then overfits |

The epoch change matters most. 25 epochs of full fine-tuning on ~121 participants will overfit hard —
full fine-tuning reaches its best validation loss in roughly 3-8 epochs on a corpus this size. Early
stopping (patience 3) handles it, and the best checkpoint is still selected on validation loss as
before.

### One warning worth reading before you run this

You asked for nothing to be frozen, and that is what the config does. But **every official
wav2vec2/XLS-R fine-tuning recipe freezes the CNN feature extractor** (`freeze_feature_encoder()`) —
including HuggingFace's own examples and the original fairseq recipe. The convolutional front-end is
known to be unstable under gradient updates and can collapse, taking the run with it.

`FREEZE_FEATURE_EXTRACTOR = False` in the config cell honours your instruction. If training loss goes
to `nan`, or validation accuracy sits at chance and will not move, **flip it to `True` first** — it
unfreezes 24 transformer layers either way, so it is still a full fine-tune of the part that matters,
and it is the single most likely fix. The config cell prints a reminder of this.

### What removing ADHD changes, and what it does not

| | 3-class (paper) | This notebook (binary) |
|---|---|---|
| Classes | ASD, ADHD, TD | **ASD, TD** |
| Chance accuracy | 33.3% | **50.0%** |
| `N_CLASSES` / output layer | 3 | **2** |
| Comparable to the paper's 0.769? | yes | **no — different task AND different encoder** |

That last row now has two reasons in it. An accuracy of, say, 0.78 here is neither "matching the
paper" nor "beating WavLM" unless you run both on this same binary task and compare them directly.
Section 9 labels the paper's number as a reference point only.

**Dataset layout expected** (adjust `DATA_ROOT` in the config cell if yours differs):

```
/kaggle/input/<dataset-name>/dataset/
├── Asymmetries/            # .cha CHAT transcripts (child-speech timestamps)
│   ├── SK-ASD/   asd01.cha  ...
│   └── SK-TD/    td01.cha   ... td38.cha
├── SK-ASD/                  # matching audio, same basenames
│   ├── asd01.mp3 ...
└── SK-TD/
    └── td01.mp3 ...
```

An `SK-ADHD` folder may sit alongside these — it is ignored. Section 2 asserts that exactly ASD and TD
are active, so a stray ADHD folder cannot quietly turn this back into a 3-class run.

> **Known deviations from the paper to keep in mind when comparing numbers:**
> - The encoder is XLS-R, not WavLM-Base-Plus — a different model of a different size.
> - Nothing is frozen here; the paper's recipe fine-tuned far less.
> - The task is binary, not 3-class.
> - Audio here is `.mp3`, not the original Olympus `.wma` — lossy re-encoding can shift
>   jitter/shimmer/HNR slightly.
> - `USE_CV=True` runs 5-fold participant-level CV rather than the paper's single split.


## 0. Setup

In [ ]:
# Install packages not preinstalled on Kaggle (safe to re-run)
!pip install -q praat-parselmouth pylangacq soundfile librosa audiomentations


In [ ]:
import os, re, json, math, random, warnings, itertools
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew, kurtosis

import librosa
import soundfile as sf
import parselmouth
from parselmouth.praat import call as praat_call

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("Environment ready.")


## 0b. GPU verification

Kaggle notebooks default to **CPU only** — this has to be turned on by hand, this file can't do it for
you: open **Settings** (the ⋮ menu / gear icon in the right sidebar) → **Accelerator** → pick
**GPU T4 x2** (or **GPU P100**) → **Save**, then re-run.

**This notebook needs a GPU more than the WavLM one did.** Full fine-tuning of a 315 M-parameter
encoder is not merely slow on CPU, it is impossible in any practical sense. Sections 1-4 (CHAT parsing
and audio cropping) run fine on CPU; Section 5 onward will refuse to start without CUDA.

A note on memory: XLS-R 300m fully unfrozen needs roughly **5 GB for weights, gradients and AdamW
state before a single sample is loaded** (315 M x 4 bytes each for weights and grads, x 8 for Adam's
two moments). The config in Section 5 is tuned for a 15 GB T4 — batch 4, gradient checkpointing, fp16.
On a 16 GB P100 it also fits, but P100 has no usable fp16 tensor cores, so expect it to be slower than
a T4 here despite the newer-looking spec sheet.


In [ ]:
import subprocess
import torch

cuda_ok = torch.cuda.is_available()
print(f"PyTorch: {torch.__version__} | CUDA available: {cuda_ok}")

if cuda_ok:
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {props.name} | {props.total_memory / 1e9:.1f} GB total")
    try:
        smi = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,memory.used,utilization.gpu",
             "--format=csv"],
            capture_output=True, text=True, check=True,
        )
        print(smi.stdout)
    except Exception as e:
        print(f"  (nvidia-smi unavailable: {e})")
else:
    print(
        "\n*** No GPU detected — accelerator is off (or set to 'None'). ***\n"
        "On Kaggle: Settings (⋮ menu, top right) -> Accelerator -> 'GPU T4 x2' or 'GPU P100' -> Save,\n"
        "then Run All again. Sections 1-7 (conventional acoustic-feature pipeline) still work on CPU,\n"
        "but Section 8 (WavLM-Base-Plus fine-tuning, 25 epochs) will be impractically slow without a GPU\n"
        "and cell-26 below will raise once training is about to start."
    )


## 1. Configuration\n\nAdjust `DATA_ROOT` to match your Kaggle input mount path.

In [ ]:
# ---- EDIT THIS to match your Kaggle "Add Input" mount path ----
DATA_ROOT = Path("/kaggle/input/datasets/asifshafin/asd-voice-full")   # <- change if your dataset slug differs
# (CHA_ROOT / AUDIO_ROOT are gone: the old config joined them with absolute paths, where the
# absolute operand wins and the prefix is silently discarded. Paths are built from _DATASET below.)

WORK_DIR = Path("/kaggle/working")
SEGMENTS_DIR = WORK_DIR / "segments"          # cropped child-speech clips get written here
FEATURES_CSV = WORK_DIR / "acoustic_features.csv"
SEGMENTS_DIR.mkdir(parents=True, exist_ok=True)

# class label -> (cha subfolder name, audio subfolder name, filename prefix)
# BINARY TASK: ASD vs TD only. ADHD is deliberately absent — not commented out, removed, so that
# there is no "option" left to half-enable. Everything downstream (N_CLASSES, the output layer, the
# class weights, the confusion matrix) is derived from what is in this dict, so this is the single
# place the task's class set is defined.
_DATASET = Path("/kaggle/input/datasets/asifshafin/asd-voice-full/dataset")

CLASS_CONFIG = {
    "ASD": {"cha_dir": _DATASET / "/kaggle/input/datasets/asifshafin/asd-voice-full/dataset/Asymmetries/SK-ASD", "audio_dir": _DATASET / "/kaggle/input/datasets/asifshafin/asd-voice-full/dataset/SK-ASD", "prefix": "asd"},
    "TD":  {"cha_dir": _DATASET / "/kaggle/input/datasets/asifshafin/asd-voice-full/dataset/Asymmetries/SK-TD",  "audio_dir": _DATASET / "/kaggle/input/datasets/asifshafin/asd-voice-full/dataset/SK-TD",  "prefix": "td"},
}

# What Section 2 will enforce. If an SK-ADHD folder exists in the dataset it is simply never looked
# at, and this assertion makes sure it cannot creep back in through an edit above.
EXPECTED_CLASSES = ["ASD", "TD"]

MIN_SEGMENT_SEC = 1.0     # paper: exclude child-speech segments shorter than 1s
TARGET_SR = 16000         # WavLM input rate; also used for feature extraction for consistency


## 2. Discover available classes & files

Confirms that both class folders have matched `.cha` + audio pairs, then **asserts the active class
set is exactly ASD and TD**. A missing folder or an unexpected extra class stops the run here rather
than changing the task shape silently three sections later.


In [ ]:
def discover_pairs(class_name, cfg):
    cha_dir, audio_dir, prefix = cfg["cha_dir"], cfg["audio_dir"], cfg["prefix"]
    if not cha_dir.exists() or not audio_dir.exists():
        return []
    cha_files = {p.stem: p for p in cha_dir.glob("*.cha")}
    audio_exts = (".mp3", ".wav", ".wma", ".m4a")
    audio_files = {p.stem: p for p in audio_dir.iterdir() if p.suffix.lower() in audio_exts}
    pairs = []
    for stem, cha_path in sorted(cha_files.items()):
        if stem in audio_files:
            pairs.append((stem, cha_path, audio_files[stem]))
        else:
            print(f"  [warn] {class_name}: no audio match for {cha_path.name}")
    return pairs

ACTIVE_CLASSES = {}
for cls, cfg in CLASS_CONFIG.items():
    pairs = discover_pairs(cls, cfg)
    if pairs:
        ACTIVE_CLASSES[cls] = pairs
        print(f"{cls}: {len(pairs)} matched participant(s)")
    else:
        print(f"{cls}: 0 files found — skipped (folder missing or empty)")

# Binary task: fail loudly rather than quietly training a differently-shaped model.
found = sorted(ACTIVE_CLASSES.keys())
if found != sorted(EXPECTED_CLASSES):
    raise ValueError(
        f"Expected exactly {sorted(EXPECTED_CLASSES)} but found {found}.\n"
        f"  - missing a class -> check the folder paths in CLASS_CONFIG above\n"
        f"  - an extra class  -> this notebook is the BINARY ASD-vs-TD variant; use the 3-class "
        f"notebook instead of adding a class here."
    )

N_CLASSES = len(ACTIVE_CLASSES)
print(f"\nBinary mode confirmed: {found}  (chance accuracy = {1 / N_CLASSES:.1%})")
print("ADHD is not part of this task — an SK-ADHD folder, if present, is ignored.")


## 3. Exploratory data analysis — participant counts & demographics

Reproduces the demographic summary style of the paper's Section 3.1 (participant counts; age/sex if
present in the `.cha` `@ID` header lines).


In [ ]:
ID_LINE_RE = re.compile(r"^@ID:\s*(.*)$")

def parse_id_headers(cha_path):
    """Parse CHAT @ID header lines. Returns list of dicts (one per participant/tier declared)."""
    records = []
    with open(cha_path, encoding="utf-8", errors="replace") as f:
        for line in f:
            m = ID_LINE_RE.match(line.strip())
            if not m:
                continue
            fields = m.group(1).split("|")
            # CHAT @ID order: language|corpus|code|age|sex|group|SES|role|education|custom
            while len(fields) < 9:
                fields.append("")
            records.append({
                "language": fields[0], "corpus": fields[1], "code": fields[2],
                "age": fields[3], "sex": fields[4], "group": fields[5],
                "role": fields[7],
            })
    return records

def child_record(cha_path):
    for rec in parse_id_headers(cha_path):
        if "Target_Child" in rec["role"] or rec["role"] == "Child":
            return rec
    return None

demo_rows = []
for cls, pairs in ACTIVE_CLASSES.items():
    for stem, cha_path, audio_path in pairs:
        rec = child_record(cha_path)
        demo_rows.append({
            "class": cls, "participant": stem,
            "age_raw": rec["age"] if rec else None,
            "sex": rec["sex"] if rec else None,
        })
demo_df = pd.DataFrame(demo_rows)

def parse_chat_age_to_years(age_str):
    # CHAT age format: e.g. "9;3.15" = 9 years 3 months 15 days
    if not age_str:
        return np.nan
    m = re.match(r"(\d+);(\d+)?", age_str)
    if not m:
        return np.nan
    years = int(m.group(1))
    months = int(m.group(2)) if m.group(2) else 0
    return years + months / 12

demo_df["age_years"] = demo_df["age_raw"].apply(parse_chat_age_to_years)

summary = demo_df.groupby("class").agg(
    n=("participant", "count"),
    mean_age=("age_years", "mean"),
    pct_male=("sex", lambda s: (s.str.upper() == "M").mean() * 100 if s.notna().any() else np.nan),
).round(2)
display(summary)

fig, ax = plt.subplots(figsize=(5, 4))
sns.barplot(x=summary.index, y="n", data=summary.reset_index(), ax=ax, palette="Set2")
ax.set_title("Participants per class")
ax.set_ylabel("N children")
plt.tight_layout()
plt.savefig(WORK_DIR / "fig_class_counts.png", dpi=150)
plt.show()


## 4. Preprocessing — crop child-only speech from `.cha` timestamps

CHAT files mark utterance-level timing with a bullet code `\x15start_end\x15` (milliseconds) appended
to the speaker's tier line. We isolate the child speaker's tier (role == Target_Child, tier code found
from the `@ID` header, typically `CHI`), collect its timestamped intervals, and crop those spans out of
the full session audio — matching the paper's Section 3.2.


In [ ]:
TIER_RE = re.compile(r"^\*([A-Za-z0-9_]+):\t?(.*)$")
BULLET_TS_RE = re.compile(r"\x15(\d+)_(\d+)\x15")

def get_child_tier_code(cha_path):
    for rec in parse_id_headers(cha_path):
        if "Target_Child" in rec["role"] or rec["role"] == "Child":
            return rec["code"] or "CHI"
    return "CHI"  # CHAT convention fallback

def extract_child_intervals_ms(cha_path):
    """Return list of (start_ms, end_ms) for the child speaker's utterances."""
    child_code = get_child_tier_code(cha_path)
    intervals = []
    current_tier = None
    buf = ""
    with open(cha_path, encoding="utf-8", errors="replace") as f:
        lines = f.readlines()

    for line in lines:
        line = line.rstrip("\n")
        m = TIER_RE.match(line)
        if m:
            # flush previous buffered utterance
            if current_tier == child_code:
                for s, e in BULLET_TS_RE.findall(buf):
                    intervals.append((int(s), int(e)))
            current_tier, buf = m.group(1), m.group(2)
        elif line.startswith("%"):
            # dependent tier (e.g. %mor, %gra) - not part of the utterance text
            continue
        elif line.startswith("@"):
            if current_tier == child_code:
                for s, e in BULLET_TS_RE.findall(buf):
                    intervals.append((int(s), int(e)))
            current_tier, buf = None, ""
        else:
            # continuation of a wrapped utterance line
            if current_tier is not None:
                buf += " " + line
    if current_tier == child_code:
        for s, e in BULLET_TS_RE.findall(buf):
            intervals.append((int(s), int(e)))
    return intervals

def crop_child_segments(class_name, stem, cha_path, audio_path, out_dir):
    intervals = extract_child_intervals_ms(cha_path)
    if not intervals:
        return []
    y, sr = librosa.load(str(audio_path), sr=TARGET_SR, mono=True)
    out_paths = []
    for i, (s_ms, e_ms) in enumerate(intervals):
        if e_ms <= s_ms:
            continue
        dur_s = (e_ms - s_ms) / 1000.0
        if dur_s < MIN_SEGMENT_SEC:
            continue
        s_samp, e_samp = int(s_ms / 1000 * sr), int(e_ms / 1000 * sr)
        e_samp = min(e_samp, len(y))
        if e_samp <= s_samp:
            continue
        clip = y[s_samp:e_samp]
        out_path = out_dir / f"{stem}_{i:03d}.wav"
        sf.write(out_path, clip, sr)
        out_paths.append(out_path)
    return out_paths

# Run cropping for every participant in every active class
segment_index = []  # rows: class, participant, segment_path, duration_s
for cls, pairs in ACTIVE_CLASSES.items():
    cls_dir = SEGMENTS_DIR / cls
    cls_dir.mkdir(parents=True, exist_ok=True)
    for stem, cha_path, audio_path in pairs:
        seg_paths = crop_child_segments(cls, stem, cha_path, audio_path, cls_dir)
        for sp in seg_paths:
            dur = librosa.get_duration(path=str(sp))
            segment_index.append({"class": cls, "participant": stem, "segment_path": str(sp), "duration_s": dur})
        if not seg_paths:
            print(f"  [warn] no valid (>= {MIN_SEGMENT_SEC}s) child segments extracted for {cls}/{stem} "
                  f"— check .cha bullet-timestamp coverage for this file")

seg_df = pd.DataFrame(segment_index)
print(f"Total child-speech segments extracted: {len(seg_df)}")
seg_df.groupby("class").agg(n_segments=("segment_path", "count"),
                             n_participants=("participant", "nunique"),
                             mean_dur_s=("duration_s", "mean")).round(2)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.histplot(data=seg_df, x="duration_s", hue="class", bins=30, element="step", stat="density", common_norm=False, ax=ax)
ax.set_title("Child-speech segment duration distribution by class")
ax.set_xlabel("Duration (s)")
plt.tight_layout()
plt.savefig(WORK_DIR / "fig_segment_durations.png", dpi=150)
plt.show()


## 5. Deep learning pipeline — XLS-R 300m, fully fine-tuned

Fine-tunes **`facebook/wav2vec2-xls-r-300m`** — all 24 transformer layers, the CNN feature extractor,
and the head — with a learned weighted sum over layer hidden states, masked mean pooling over time, a
dropout + linear classification head, weighted cross-entropy with label smoothing, and the paper's
augmentation recipe (speed perturbation, time dropout, gain jitter).

Added for full fine-tuning: fp16 mixed precision, gradient checkpointing, gradient accumulation,
linear warmup + decay, gradient clipping, and early stopping. See the overview at the top for why each
one is here.


In [ ]:
# transformers >= 4.30 has the Wav2Vec2 gradient-checkpointing API used below
!pip install -q "transformers>=4.30" accelerate


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is off for this session. Enable it via Settings -> Accelerator -> 'GPU T4 x2' or "
        "'GPU P100' -> Save, then re-run from Section 0b onward. Full fine-tuning of a 315M-parameter "
        "XLS-R encoder on CPU is not practical in any sense."
    )
DEVICE = torch.device("cuda")
print(f"Using device: {DEVICE} ({torch.cuda.get_device_name(0)})")

# ===========================================================================
# ENCODER
# ===========================================================================
# facebook/wav2vec2-xls-r-300m: 315M params, 24 layers, hidden 1024, pretrained on 436k hours
# across 128 languages -- Bangla among them, via the CommonVoice and VoxLingua107 portions.
#
# The 1b and 2b XLS-R checkpoints exist and are stronger, but neither fits a full fine-tune on a
# 15GB T4 -- 1b alone needs ~16GB for weights + grads + Adam state before any activation.
XLSR_CHECKPOINT = "facebook/wav2vec2-xls-r-300m"

# ===========================================================================
# POOLING
# ===========================================================================
#   "weighted_sum"  learned softmax weight per layer over all 25 hidden states (embeddings + 24
#                   layers), then masked mean over time. Every layer is used and every layer gets
#                   gradient -- which is what "freeze nothing" has to mean. Standard SUPERB/s3prl
#                   recipe for XLS-R. DEFAULT.
#   "fixed_layer"   pool hidden_states[XLSR_LAYER] only, matching the WavLM notebook's structure.
#                   NOTE: layers ABOVE XLSR_LAYER then receive zero gradient regardless of
#                   requires_grad, so "nothing is frozen" stops being true in any useful sense.
#   "last_layer"    pool the final layer. All layers contribute, no extra parameters.
POOL_MODE = "weighted_sum"
XLSR_LAYER = 12          # only used when POOL_MODE == "fixed_layer" (XLS-R has 24 layers, not 12)

# ===========================================================================
# FREEZING -- nothing is frozen, as requested
# ===========================================================================
# Every official wav2vec2/XLS-R recipe (HuggingFace's own examples, the original fairseq one)
# calls freeze_feature_encoder(). The CNN front-end is known to be unstable under gradient
# updates and can collapse the run. This is False because you asked for nothing frozen.
#
# >>> If training loss hits nan, or val accuracy sits at chance and will not move, set this to
# >>> True FIRST. All 24 transformer layers stay trainable either way.
FREEZE_FEATURE_EXTRACTOR = False

# ===========================================================================
# AUDIO
# ===========================================================================
MAX_DURATION_S = 10
MIN_DURATION_S = 1.0

# XLS-R is a feat_extract_norm="layer" model: it REQUIRES zero-mean unit-variance input.
# WavLM-Base-Plus is "group" / do_normalize=False, which is why the notebook this is derived from
# could pass raw librosa output straight in (it built a feature extractor and never called it).
# Doing that here does not raise -- it just trains badly. Read the real value off the config
# rather than assuming it.
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(XLSR_CHECKPOINT)
DO_NORMALIZE = bool(getattr(feature_extractor, "do_normalize", True))
print(f"\nFeature extractor: do_normalize={DO_NORMALIZE}, "
      f"return_attention_mask={getattr(feature_extractor, 'return_attention_mask', None)}, "
      f"sampling_rate={feature_extractor.sampling_rate}")
if feature_extractor.sampling_rate != TARGET_SR:
    raise ValueError(
        f"Checkpoint expects {feature_extractor.sampling_rate} Hz but TARGET_SR is {TARGET_SR}. "
        f"Set TARGET_SR in Section 1 to match and re-run the cropping section."
    )

# ===========================================================================
# TRAINING -- tuned for a full fine-tune on a 15GB T4
# ===========================================================================
BATCH_SIZE = 4           # per step. 315M unfrozen + 10s audio + 24 layers does not fit at 8.
GRAD_ACCUM_STEPS = 2     # 4 x 2 = effective batch 8, matching the WavLM notebook
N_EPOCHS = 10            # full FT converges in ~3-8 epochs here; 25 would just overfit
EARLY_STOP_PATIENCE = 3  # epochs without val-loss improvement before stopping this fold
LR = 1e-5                # encoder LR for full fine-tuning
HEAD_LR = 1e-3           # the randomly-initialised head needs far more than the pretrained encoder
WARMUP_RATIO = 0.10      # wav2vec2-family full FT is unstable without warmup
MAX_GRAD_NORM = 1.0      # same reason
LABEL_SMOOTHING = 0.05
USE_AMP = True           # fp16. Roughly 2x on a T4 and a large part of why 315M fits at all.
GRADIENT_CHECKPOINTING = True   # ~40% less activation memory for ~30% more time

EFFECTIVE_BATCH = BATCH_SIZE * GRAD_ACCUM_STEPS

# ---- class set (unchanged from the WavLM notebook) ----
CLASSES_SORTED = sorted(seg_df["class"].unique())
if CLASSES_SORTED != sorted(EXPECTED_CLASSES):
    raise ValueError(
        f"Segments cover {CLASSES_SORTED} but the task is {sorted(EXPECTED_CLASSES)}. "
        f"A class discovered in Section 2 produced no usable segments in Section 4 — check the "
        f"[warn] lines there for a participant whose .cha has no child-speech timestamps."
    )
N_CLASSES = len(CLASSES_SORTED)
label2id = {c: i for i, c in enumerate(CLASSES_SORTED)}
id2label = {i: c for c, i in label2id.items()}

print(f"\nClasses: {CLASSES_SORTED}  (binary, chance = {1 / N_CLASSES:.1%})")
print(f"Encoder: {XLSR_CHECKPOINT}")
print(f"Pooling: {POOL_MODE}" + (f" (layer {XLSR_LAYER})" if POOL_MODE == "fixed_layer" else ""))
print(f"Batch {BATCH_SIZE} x {GRAD_ACCUM_STEPS} accum = {EFFECTIVE_BATCH} effective | "
      f"{N_EPOCHS} epochs max, early stop after {EARLY_STOP_PATIENCE}")
print(f"AMP fp16: {USE_AMP} | gradient checkpointing: {GRADIENT_CHECKPOINTING}")
print(f"Feature extractor (CNN front-end) frozen: {FREEZE_FEATURE_EXTRACTOR}")
if not FREEZE_FEATURE_EXTRACTOR:
    print("  ^ nothing is frozen, as requested. If the loss goes to nan or val accuracy is stuck")
    print("    at chance, set FREEZE_FEATURE_EXTRACTOR = True -- that is the usual cause and every")
    print("    official wav2vec2/XLS-R recipe does it. The 24 transformer layers stay trainable.")


In [ ]:
# Evaluation protocol: single held-out split (the paper's DL setup) or 5-fold CV.
#
# The paper used 10-fold nested CV for the CLASSICAL classifiers (Tables 1-2) but only a single
# stratified train/val/test split for the deep model (Table 3). USE_CV=True is therefore a
# deliberate deviation — a stricter evaluation, since a single test fold here is only ~18 of the
# 121 participants and one child landing differently moves accuracy by several points.
USE_CV = True
N_FOLDS = 5
VAL_FRACTION = 0.15   # carved out of each fold's TRAINING participants, for checkpoint selection

from sklearn.model_selection import StratifiedKFold

# One row per participant. EVERY split below is made on this table, never on segments: a child's
# clips are near-duplicates of each other, so splitting by segment would put the same voice in
# train and test and inflate the score badly.
participants_df = seg_df.drop_duplicates("participant")[["participant", "class"]].reset_index(drop=True)
print(f"Participants: {len(participants_df)} total")
print(participants_df["class"].value_counts().to_string())

def subset_by_participants(df, part_df):
    return df[df["participant"].isin(part_df["participant"])].reset_index(drop=True)

def describe(name, tr, va, te):
    print(f"  {name}: train {len(tr):>5} seg / {tr['participant'].nunique():>3} kids | "
          f"val {len(va):>4} seg / {va['participant'].nunique():>2} kids | "
          f"test {len(te):>4} seg / {te['participant'].nunique():>2} kids")

if USE_CV:
    smallest = participants_df["class"].value_counts().min()
    if smallest < N_FOLDS:
        raise ValueError(
            f"N_FOLDS={N_FOLDS} but the smallest class has only {smallest} participants. "
            f"Stratified folds need at least one participant per class per fold — lower N_FOLDS "
            f"to {smallest} or less, or set USE_CV=False."
        )
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
    FOLDS = []
    for fold, (fit_idx, test_idx) in enumerate(
            skf.split(participants_df, participants_df["class"]), start=1):
        fit_p = participants_df.iloc[fit_idx]
        test_p = participants_df.iloc[test_idx]
        # carve a validation set out of this fold's training participants (never out of test)
        tr_p, va_p = train_test_split(
            fit_p, test_size=VAL_FRACTION, stratify=fit_p["class"], random_state=RANDOM_SEED)
        FOLDS.append({
            "fold": fold,
            "train_df": subset_by_participants(seg_df, tr_p),
            "val_df": subset_by_participants(seg_df, va_p),
            "test_df": subset_by_participants(seg_df, test_p),
        })

    print(f"\n{N_FOLDS}-fold participant-level CV — every participant is tested exactly once:")
    for f in FOLDS:
        describe(f"fold {f['fold']}", f["train_df"], f["val_df"], f["test_df"])

    # sanity: no participant may appear in two different test folds, and none may be in both
    # train and test within a fold
    seen = set()
    for f in FOLDS:
        te = set(f["test_df"]["participant"])
        assert not (seen & te), "a participant landed in two test folds"
        seen |= te
        assert not (set(f["train_df"]["participant"]) & te), "participant leaked train->test"
        assert not (set(f["val_df"]["participant"]) & te), "participant leaked val->test"
    assert seen == set(participants_df["participant"]), "not every participant was tested"
    print(f"  leakage checks passed; {len(seen)} participants each tested exactly once")
else:
    train_p, temp_p = train_test_split(
        participants_df, test_size=0.3, stratify=participants_df["class"], random_state=RANDOM_SEED)
    val_p, test_p = train_test_split(
        temp_p, test_size=0.5, stratify=temp_p["class"], random_state=RANDOM_SEED)
    FOLDS = [{
        "fold": 1,
        "train_df": subset_by_participants(seg_df, train_p),
        "val_df": subset_by_participants(seg_df, val_p),
        "test_df": subset_by_participants(seg_df, test_p),
    }]
    print("\nSingle held-out split (paper's DL protocol):")
    describe("split", FOLDS[0]["train_df"], FOLDS[0]["val_df"], FOLDS[0]["test_df"])

# kept so any cell that still refers to the single-split names works
train_df, val_df, test_df = FOLDS[0]["train_df"], FOLDS[0]["val_df"], FOLDS[0]["test_df"]

In [ ]:
def speed_perturb(y, sr, factor_range=(0.9, 1.1)):
    factor = np.random.uniform(*factor_range)
    return librosa.effects.time_stretch(y, rate=factor)

def gain_jitter(y, db_range=(-6, 6)):
    gain_db = np.random.uniform(*db_range)
    return y * (10 ** (gain_db / 20))

def time_dropout(y, sr, max_drop_s=0.5, n_drops=2):
    y = y.copy()
    for _ in range(n_drops):
        drop_len = int(np.random.uniform(0, max_drop_s) * sr)
        if drop_len == 0 or len(y) <= drop_len:
            continue
        start = np.random.randint(0, len(y) - drop_len)
        y[start:start + drop_len] = 0.0
    return y

def normalize_waveform(y, valid_len=None):
    """Zero-mean, unit-variance per utterance -- what XLS-R's feat_extract_norm='layer' front-end
    expects, and what Wav2Vec2FeatureExtractor(do_normalize=True) does internally.

    `valid_len` matters: statistics are computed over the REAL audio only. Including the zero
    padding would pull the mean toward zero and shrink the variance by however much of the window
    happens to be padding -- a clip-length-dependent distortion, and clip lengths differ by class
    in this corpus, so it would not be a neutral one.

    Applied AFTER augmentation on purpose: gain_jitter multiplies the waveform, and normalising
    afterwards would divide that scaling straight back out, silently turning that augmentation
    into a no-op."""
    if not DO_NORMALIZE:
        return y
    seg = y if valid_len is None else y[:valid_len]
    if len(seg) == 0:
        return y
    # Matches Wav2Vec2FeatureExtractor.zero_mean_unit_var_norm exactly, including the
    # sqrt(var + 1e-7) form rather than (std + 1e-7).
    normed = (y - seg.mean()) / np.sqrt(seg.var() + 1e-7)
    # HF re-zeroes the padding AFTER normalising. Without this the padded tail becomes the
    # constant -mean/std rather than silence -- a nonzero DC offset that the CNN front-end does
    # see (the attention mask only protects attention and pooling, not the convolutions), and
    # whose size depends on how much of the window is padding, which varies by clip length and
    # therefore by class in this corpus.
    if valid_len is not None and valid_len < len(normed):
        normed[valid_len:] = 0.0
    return normed

def pad_or_crop(y, sr, target_s=MAX_DURATION_S, random_crop=True):
    """Returns (fixed_length_array, valid_length).

    valid_length = how much of the array is real audio rather than zero-padding, so pooling can
    exclude the padding instead of averaging it in. With clip lengths that differ by class (TD
    clips run shorter than ASD ones in this corpus), unmasked pooling dilutes the shorter class
    more — a class-dependent handicap, not a neutral one.

    random_crop=False makes the crop deterministic. Inference MUST use that: predict_proba is
    called twice per occlusion measurement (clean vs occluded), and a random window each call
    means the difference measures which window was drawn, not the occlusion."""
    target_len = int(target_s * sr)
    if len(y) >= target_len:
        if random_crop and len(y) > target_len:
            start = np.random.randint(0, len(y) - target_len + 1)
        else:
            start = 0
        return y[start:start + target_len], target_len
    return np.pad(y, (0, target_len - len(y))), len(y)

class SpeechDataset(Dataset):
    def __init__(self, df, augment=False):
        self.df = df.reset_index(drop=True)
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        y, sr = librosa.load(row["segment_path"], sr=TARGET_SR, mono=True)
        if len(y) < int(MIN_DURATION_S * sr):
            y = np.pad(y, (0, int(MIN_DURATION_S * sr) - len(y)))
        if self.augment:
            if np.random.rand() < 0.5:
                y = speed_perturb(y, sr)
            if np.random.rand() < 0.5:
                y = gain_jitter(y)
            if np.random.rand() < 0.5:
                y = time_dropout(y, sr)
        y, valid_len = pad_or_crop(y, sr, random_crop=self.augment)
        # XLS-R requires normalised input -- see normalize_waveform's docstring for why this is
        # here and not before the augmentations.
        y = normalize_waveform(y, valid_len)
        label = label2id[row["class"]]
        return {
            "input_values": torch.tensor(y, dtype=torch.float32),
            "valid_length": valid_len,
            "label": label,
        }

def collate_fn(batch):
    input_values = torch.stack([b["input_values"] for b in batch])
    lengths = torch.tensor([b["valid_length"] for b in batch], dtype=torch.long)
    # 1 where the sample is real audio, 0 where it is padding. The model's own
    # _get_feature_vector_attention_mask downsamples this to the encoder's time axis, so masked
    # pooling averages over real speech only. XLS-R's config sets return_attention_mask=True --
    # unlike the wav2vec2-base checkpoints, it was pretrained WITH a mask, so passing one is
    # correct here rather than merely harmless.
    attention_mask = (torch.arange(input_values.shape[1]).unsqueeze(0) < lengths.unsqueeze(1)).long()
    labels = torch.tensor([b["label"] for b in batch], dtype=torch.long)
    return {"input_values": input_values, "attention_mask": attention_mask, "labels": labels}

def make_loaders(train_df, val_df, test_df):
    """Built per fold rather than once, so each fold gets loaders over its own participants."""
    train_ds = SpeechDataset(train_df, augment=True)
    val_ds = SpeechDataset(val_df, augment=False)
    test_ds = SpeechDataset(test_df, augment=False)
    # Evaluation has no backward pass, so it can use a larger batch for free. num_workers=2 keeps
    # the mp3 decode + resample off the training step's critical path.
    eval_bs = BATCH_SIZE * 2
    return (
        DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn,
                   num_workers=2, pin_memory=True, drop_last=False),
        DataLoader(val_ds, batch_size=eval_bs, shuffle=False, collate_fn=collate_fn,
                   num_workers=2, pin_memory=True),
        DataLoader(test_ds, batch_size=eval_bs, shuffle=False, collate_fn=collate_fn,
                   num_workers=2, pin_memory=True),
    )


In [ ]:
class XLSRClassifier(nn.Module):
    """XLS-R 300m + pooling + linear head. Nothing is frozen unless
    FREEZE_FEATURE_EXTRACTOR is True."""

    def __init__(self, checkpoint, n_classes, pool_mode=POOL_MODE, fixed_layer=XLSR_LAYER,
                 dropout=0.1, freeze_feature_extractor=FREEZE_FEATURE_EXTRACTOR,
                 gradient_checkpointing=GRADIENT_CHECKPOINTING):
        super().__init__()
        self.encoder = Wav2Vec2Model.from_pretrained(checkpoint)

        # LayerDrop MUST be off whenever a specific hidden-state INDEX is read.
        # In train() mode the encoder randomly skips layers and does not record a hidden state for
        # a skipped one, so output_hidden_states returns a SHORTER tuple and every index above the
        # skipped layer shifts down by one. hidden_states[12] then silently reads layer 11, or 10,
        # on some batches, while eval mode (LayerDrop inactive) always reads the true layer 12.
        # weighted_sum is affected just as badly: the softmax weights are learned per POSITION, so
        # a shifted tuple means weight i is applied to a different layer than it was learned for.
        self.encoder.config.layerdrop = 0.0
        self.encoder.encoder.config.layerdrop = 0.0

        # SpecAugment on the encoder's latent features. XLS-R ships mask_time_prob=0.05; it is
        # active in train() mode only and is a genuine regulariser on a corpus this small, so it
        # is left on deliberately rather than by omission.
        self.encoder.config.apply_spec_augment = True

        if gradient_checkpointing:
            # Recomputes activations during backward instead of storing them: ~40% less activation
            # memory for ~30% more time. This is what lets 24 unfrozen layers fit on a T4.
            # use_reentrant=False matters: the reentrant implementation needs at least one
            # input with requires_grad=True reaching each checkpointed block, and with a frozen
            # CNN front-end that condition can silently fail, producing zero gradients for the
            # whole encoder with no error raised. Older transformers has no kwargs argument, so
            # fall back rather than hard-depend on the version.
            try:
                self.encoder.gradient_checkpointing_enable(
                    gradient_checkpointing_kwargs={"use_reentrant": False})
            except TypeError:
                self.encoder.gradient_checkpointing_enable()
            # use_cache and gradient checkpointing are mutually exclusive; the encoder has no cache
            # in this configuration but newer transformers warns loudly about it every step.
            self.encoder.config.use_cache = False

        self.pool_mode = pool_mode
        self.n_layers = self.encoder.config.num_hidden_layers
        self.n_states = self.n_layers + 1          # embeddings output + one per layer
        hidden_size = self.encoder.config.hidden_size

        if pool_mode == "fixed_layer":
            self.fixed_layer = fixed_layer if fixed_layer >= 0 else self.n_states + fixed_layer
            if not 0 <= self.fixed_layer < self.n_states:
                raise ValueError(
                    f"XLSR_LAYER={fixed_layer} is out of range: this encoder has {self.n_layers} "
                    f"layers, so valid hidden-state indices are 0..{self.n_states - 1}."
                )
        elif pool_mode == "weighted_sum":
            # One learnable scalar per hidden state, softmaxed so the weights sum to 1. Initialised
            # uniform (all zeros -> softmax gives 1/n_states each), so the first forward pass is a
            # plain average over layers rather than an arbitrary preference.
            self.layer_weights = nn.Parameter(torch.zeros(self.n_states))
        elif pool_mode != "last_layer":
            raise ValueError(
                f"POOL_MODE must be 'weighted_sum', 'fixed_layer' or 'last_layer', got {pool_mode!r}"
            )

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, n_classes)

        if freeze_feature_extractor:
            self.encoder.freeze_feature_encoder()

        self._report(freeze_feature_extractor, gradient_checkpointing)

    def _report(self, froze_fe, grad_ckpt):
        n_total = sum(p.numel() for p in self.parameters())
        n_trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"XLS-R: {self.n_layers} layers, hidden {self.encoder.config.hidden_size}")
        if self.pool_mode == "weighted_sum":
            print(f"  pooling: learned weighted sum over all {self.n_states} hidden states "
                  f"-> masked mean over time")
            print(f"           every layer contributes and every layer receives gradient")
        elif self.pool_mode == "fixed_layer":
            print(f"  pooling: hidden_states[{self.fixed_layer}] -> masked mean over time")
            above = self.n_states - 1 - self.fixed_layer
            if above > 0:
                print(f"  *** {above} layer(s) above the pooling point receive ZERO gradient no ***")
                print(f"  *** matter what requires_grad says -- they are computed every forward  ***")
                print(f"  *** pass and contribute nothing. Use POOL_MODE='weighted_sum' if you   ***")
                print(f"  *** actually want all {self.n_layers} layers trained.                          ***")
        else:
            print(f"  pooling: final layer -> masked mean over time")
        print(f"  CNN feature extractor frozen: {froze_fe}")
        print(f"  gradient checkpointing: {grad_ckpt}")
        print(f"  trainable params: {n_trainable:,} / {n_total:,} "
              f"({100 * n_trainable / n_total:.1f}%)")
        if n_trainable == n_total:
            print(f"  -> nothing is frozen.")

    def forward(self, input_values, attention_mask=None):
        need_states = self.pool_mode in ("weighted_sum", "fixed_layer")
        outputs = self.encoder(input_values, attention_mask=attention_mask,
                               output_hidden_states=need_states)

        if self.pool_mode == "last_layer":
            hidden = outputs.last_hidden_state
        else:
            states = outputs.hidden_states
            # If this ever fails, LayerDrop is back on (see __init__) and the indices below no
            # longer mean what they say.
            if len(states) != self.n_states:
                raise RuntimeError(
                    f"Got {len(states)} hidden states, expected {self.n_states}. LayerDrop is "
                    f"active, so layer indices and learned layer weights are misaligned."
                )
            if self.pool_mode == "fixed_layer":
                hidden = states[self.fixed_layer]
            else:
                w = torch.softmax(self.layer_weights, dim=0)          # (n_states,)
                stacked = torch.stack(states, dim=0)                   # (n_states, B, T, H)
                hidden = (stacked * w.view(-1, 1, 1, 1)).sum(0)        # (B, T, H)

        if attention_mask is not None:
            feat_mask = self.encoder._get_feature_vector_attention_mask(hidden.shape[1], attention_mask)
            mask = feat_mask.unsqueeze(-1).to(hidden.dtype)
            pooled = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-6)
        else:
            pooled = hidden.mean(1)
        pooled = self.dropout(pooled)
        return self.classifier(pooled)

    def layer_weight_summary(self):
        """Softmaxed layer weights, for inspecting which depths the task actually used.
        Returns None unless POOL_MODE == 'weighted_sum'."""
        if self.pool_mode != "weighted_sum":
            return None
        with torch.no_grad():
            return torch.softmax(self.layer_weights.detach().cpu(), dim=0).numpy()


def build_model_and_optimizer(train_df, steps_per_epoch):
    """A FRESH model + optimizer + scheduler per fold.

    This must not be hoisted out of the fold loop: reusing one model across folds would let fold 2
    start from weights already trained on fold 1's data — which includes fold 2's test
    participants. Every fold's score after the first would be contaminated.
    """
    model = XLSRClassifier(XLSR_CHECKPOINT, n_classes=N_CLASSES).to(DEVICE)

    # Class weights from THIS fold's training distribution (paper: weighted CE for imbalance)
    class_counts = train_df["class"].value_counts()
    weights = torch.tensor([1.0 / class_counts[c] for c in CLASSES_SORTED], dtype=torch.float32)
    weights = weights / weights.sum() * N_CLASSES
    criterion = nn.CrossEntropyLoss(weight=weights.to(DEVICE), label_smoothing=LABEL_SMOOTHING)

    # Three parameter groups. The head is randomly initialised and needs a much larger LR than the
    # pretrained encoder; the layer weights are also fresh, and they are 25 scalars whose softmax
    # has to move meaningfully within a handful of epochs, so they get the head's LR too.
    head_names = ("classifier", "layer_weights")
    head_params, enc_params = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (head_params if n.startswith(head_names) else enc_params).append(p)

    param_groups = []
    if head_params:
        param_groups.append({"params": head_params, "lr": HEAD_LR})
    if enc_params:
        param_groups.append({"params": enc_params, "lr": LR})
    optimizer = torch.optim.AdamW(param_groups, weight_decay=0.01)

    # Linear warmup then linear decay, counted in OPTIMIZER steps (not batches) so gradient
    # accumulation does not silently stretch the schedule by GRAD_ACCUM_STEPS.
    total_steps = max(1, steps_per_epoch * N_EPOCHS)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(WARMUP_RATIO * total_steps),
        num_training_steps=total_steps,
    )
    return model, criterion, optimizer, scheduler


In [ ]:
import math, time, gc

# fp16 on a T4. bf16 would be preferable numerically but Turing (sm_75) has no bf16 hardware --
# torch.cuda.is_bf16_supported() can answer True there via emulation, which is slower than fp32,
# so the check is on compute capability instead of that flag.
_major, _minor = torch.cuda.get_device_capability()
AMP_DTYPE = torch.bfloat16 if _major >= 8 else torch.float16
AMP_ENABLED = USE_AMP
print(f"AMP: {'on, ' + str(AMP_DTYPE).replace('torch.', '') if AMP_ENABLED else 'off (fp32)'} "
      f"on sm_{_major}{_minor}")


def train_one_epoch(model, criterion, optimizer, scheduler, scaler, loader):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    optimizer.zero_grad(set_to_none=True)
    n_batches = len(loader)

    for step, batch in enumerate(loader):
        input_values = batch["input_values"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        labels = batch["labels"].to(DEVICE, non_blocking=True)

        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=AMP_ENABLED):
            logits = model(input_values, attention_mask=attention_mask)
        # Loss in fp32, outside autocast. Cheap, and it removes the most common source of a nan
        # during an AMP conversion.
        logits = logits.float()
        loss = criterion(logits, labels)

        # Scale down so accumulated gradients average rather than sum -- otherwise the effective
        # LR is GRAD_ACCUM_STEPS times what the config says.
        scaled = loss / GRAD_ACCUM_STEPS
        if scaler is not None:
            scaler.scale(scaled).backward()
        else:
            scaled.backward()

        is_last = (step + 1) == n_batches
        if (step + 1) % GRAD_ACCUM_STEPS == 0 or is_last:
            if scaler is not None:
                # unscale BEFORE clipping, or the clip threshold applies to loss-scaled gradients
                # and means nothing.
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                scaler.step(optimizer)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                optimizer.step()
            # Step the schedule once per OPTIMIZER step, matching how total_steps was computed.
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(-1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / max(total, 1), correct / max(total, 1)


@torch.no_grad()
def evaluate_epoch(model, criterion, loader):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for batch in loader:
        input_values = batch["input_values"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        labels = batch["labels"].to(DEVICE, non_blocking=True)
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=AMP_ENABLED):
            logits = model(input_values, attention_mask=attention_mask)
        logits = logits.float()
        loss = criterion(logits, labels)
        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(-1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / max(total, 1), correct / max(total, 1)


@torch.no_grad()
def predict_loader(model, loader):
    model.eval()
    preds, labels = [], []
    for batch in loader:
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=AMP_ENABLED):
            logits = model(batch["input_values"].to(DEVICE),
                           attention_mask=batch["attention_mask"].to(DEVICE))
        preds.extend(logits.float().argmax(-1).cpu().tolist())
        labels.extend(batch["labels"].tolist())
    return preds, labels


def train_one_fold(spec):
    """Train on one fold and return its history, its test predictions, and the trained model."""
    fold = spec["fold"]
    train_loader, val_loader, test_loader = make_loaders(
        spec["train_df"], spec["val_df"], spec["test_df"])

    steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
    model, criterion, optimizer, scheduler = build_model_and_optimizer(
        spec["train_df"], steps_per_epoch)
    # GradScaler is only needed for fp16; bf16 has fp32's exponent range and does not underflow.
    if AMP_ENABLED and AMP_DTYPE == torch.float16:
        try:
            scaler = torch.amp.GradScaler(device="cuda")      # torch >= 2.3
        except TypeError:
            scaler = torch.cuda.amp.GradScaler()              # older
    else:
        scaler = None

    ckpt = WORK_DIR / f"best_xlsr_fold{fold}.pt"
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_loss = float("inf")
    epochs_without_improvement = 0
    stopped_early = False

    for epoch in range(1, N_EPOCHS + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, criterion, optimizer, scheduler, scaler, train_loader)
        va_loss, va_acc = evaluate_epoch(model, criterion, val_loader)
        history["train_loss"].append(tr_loss); history["train_acc"].append(tr_acc)
        history["val_loss"].append(va_loss); history["val_acc"].append(va_acc)

        if not math.isfinite(tr_loss):
            raise RuntimeError(
                f"fold {fold} epoch {epoch}: training loss is {tr_loss}. Full fine-tuning of "
                f"XLS-R has diverged.\n"
                f"  First thing to try: set FREEZE_FEATURE_EXTRACTOR = True in the config cell and "
                f"re-run. The CNN front-end is the usual culprit and every official wav2vec2/XLS-R "
                f"recipe freezes it; all 24 transformer layers stay trainable.\n"
                f"  If that does not fix it: lower LR to 5e-6, or raise WARMUP_RATIO to 0.2."
            )

        improved = va_loss < best_val_loss
        if improved:
            best_val_loss = va_loss
            epochs_without_improvement = 0
            torch.save(model.state_dict(), ckpt)
        else:
            epochs_without_improvement += 1

        print(f"  fold {fold} epoch {epoch:02d}/{N_EPOCHS} | train loss {tr_loss:.4f} "
              f"acc {tr_acc:.3f} | val loss {va_loss:.4f} acc {va_acc:.3f} "
              f"| {time.time() - t0:.0f}s"
              + ("  <- best" if improved else f"  ({epochs_without_improvement}/{EARLY_STOP_PATIENCE})"))

        if epochs_without_improvement >= EARLY_STOP_PATIENCE:
            print(f"  fold {fold}: early stop at epoch {epoch} — val loss has not improved for "
                  f"{EARLY_STOP_PATIENCE} epochs. Best val loss {best_val_loss:.4f}.")
            stopped_early = True
            break

    if not stopped_early:
        print(f"  fold {fold}: ran the full {N_EPOCHS} epochs without early-stopping. If val loss "
              f"was still falling at the end, N_EPOCHS is the binding constraint — raise it.")

    # evaluate the best checkpoint, not whatever the last epoch happened to leave behind
    model.load_state_dict(torch.load(ckpt))
    preds, labels = predict_loader(model, test_loader)
    acc = accuracy_score(labels, preds)
    f1m = f1_score(labels, preds, average="macro", zero_division=0)

    lw = model.layer_weight_summary()
    if lw is not None:
        top = np.argsort(lw)[::-1][:5]
        print(f"  fold {fold} layer weights — top 5 of {len(lw)}: "
              + ", ".join(f"L{i}={lw[i]:.3f}" for i in top))

    print(f"  fold {fold} TEST accuracy {acc:.3f} | macro-F1 {f1m:.3f}\n")
    return history, preds, labels, acc, f1m, model, lw


print(f"{len(FOLDS)} fold(s): full fine-tuning XLS-R 300m {len(FOLDS)} time(s), "
      f"up to {N_EPOCHS} epochs each.")
print("This is by far the slowest part of the notebook. A 315M-parameter encoder with gradient")
print("checkpointing runs roughly 10-20x an epoch of the WavLM head-only setup — budget hours,")
print("not minutes, and expect early stopping to cut several folds short.\n")

fold_histories, fold_scores, fold_layer_weights = [], [], []
all_preds, all_labels = [], []          # pooled across folds -> every participant tested once
for spec in FOLDS:
    hist, preds, labels, acc, f1m, model, lw = train_one_fold(spec)
    fold_histories.append(hist)
    fold_scores.append({"fold": spec["fold"], "accuracy": acc, "macro_f1": f1m,
                        "n_test_segments": len(labels), "n_epochs_run": len(hist["train_loss"])})
    all_preds.extend(preds); all_labels.extend(labels)
    if lw is not None:
        fold_layer_weights.append(lw)
    # 315M params x (weights + grads + 2 Adam moments) is ~5GB per fold. The optimizer is local
    # to train_one_fold so it goes on return, but `model` is held by this loop -- without the del
    # below, fold i+1 constructs its encoder while fold i's is still resident on the GPU.
    # The LAST fold's model is deliberately kept: Sections 7 and 8 (occlusion) run against it.
    if spec is not FOLDS[-1]:
        del model, hist, preds, labels
        gc.collect()
        torch.cuda.empty_cache()

history = fold_histories[-1]            # kept for any cell expecting the single-split name
test_df = FOLDS[-1]["test_df"]          # the model left in memory is the last fold's

fold_df = pd.DataFrame(fold_scores)
if USE_CV:
    print("Per-fold results:")
    print(fold_df.round(4).to_string(index=False))
    print(f"\nAccuracy: {fold_df['accuracy'].mean():.4f} +/- {fold_df['accuracy'].std():.4f}")
    print(f"Macro-F1: {fold_df['macro_f1'].mean():.4f} +/- {fold_df['macro_f1'].std():.4f}")
    kids_per_fold = int(np.mean([f["test_df"]["participant"].nunique() for f in FOLDS]))
    print(f"\nThe +/- is the spread across folds — with ~{kids_per_fold} test participants per "
          f"fold it is usually wide, which is exactly the uncertainty a single split hides.")
    fold_df.to_csv(WORK_DIR / "cv_fold_results.csv", index=False)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for i, h in enumerate(fold_histories, start=1):
    lbl = f"fold {i}" if len(fold_histories) > 1 else None
    axes[0].plot(h["train_loss"], alpha=0.8, label=f"train {lbl}" if lbl else "train")
    axes[0].plot(h["val_loss"], "--", alpha=0.8, label=f"val {lbl}" if lbl else "val")
    axes[1].plot(h["train_acc"], alpha=0.8, label=f"train {lbl}" if lbl else "train")
    axes[1].plot(h["val_acc"], "--", alpha=0.8, label=f"val {lbl}" if lbl else "val")
axes[0].set_title("Loss (solid=train, dashed=val)"); axes[0].set_xlabel("Epoch")
axes[1].set_title("Accuracy (solid=train, dashed=val)"); axes[1].set_xlabel("Epoch")
axes[0].legend(fontsize=7); axes[1].legend(fontsize=7)
plt.tight_layout()
plt.savefig(WORK_DIR / "fig_xlsr_training_curves.png", dpi=150)
plt.show()

# Folds end at different epochs now that early stopping is on -- the lines will have different
# lengths and that is expected, not a plotting bug.
if "n_epochs_run" in fold_df.columns and fold_df["n_epochs_run"].nunique() > 1:
    print("Folds ran for different numbers of epochs (early stopping): "
          + ", ".join(f"fold {r.fold}={r.n_epochs_run}" for r in fold_df.itertuples()))
# With a 315M encoder fully unfrozen on ~121 participants, a train curve that dives while the val
# curve turns up is the expected failure mode, not a surprise. The best-val-loss checkpoint is
# what gets evaluated, so overfitting past that point costs time but not accuracy.
gap = [h["train_acc"][-1] - h["val_acc"][-1] for h in fold_histories]
print(f"Final train-val accuracy gap per fold: " + ", ".join(f"{g:+.3f}" for g in gap))
if np.mean(gap) > 0.25:
    print("  A gap this wide means the encoder is memorising participants. Options: fewer epochs,")
    print("  stronger augmentation, or POOL_MODE='fixed_layer' with a truncated encoder.")


### Which layers did the task actually use?

Only meaningful with `POOL_MODE = "weighted_sum"`. The softmaxed layer weights say where in the
encoder the ASD/TD signal was found.

Worth reading rather than skipping: for paralinguistic tasks the weight usually concentrates in the
**middle** layers, while the top layers — the ones most specialised toward the pretraining objective —
often get very little. If the weight here is concentrated around some layer *k*, that is empirical
evidence for what `XLSR_LAYER` should be if you ever want to switch to `"fixed_layer"` and truncate
the encoder above *k* for a much faster model.


In [ ]:
if fold_layer_weights:
    lw_arr = np.vstack(fold_layer_weights)          # (n_folds, n_states)
    mean_lw, std_lw = lw_arr.mean(0), lw_arr.std(0)

    fig, ax = plt.subplots(figsize=(10, 4))
    idx = np.arange(len(mean_lw))
    ax.bar(idx, mean_lw, yerr=std_lw if len(fold_layer_weights) > 1 else None,
           capsize=3, color="steelblue")
    ax.axhline(1 / len(mean_lw), color="red", ls="--", lw=1,
               label=f"uniform ({1 / len(mean_lw):.3f}) — the initialisation")
    ax.set_xlabel("hidden-state index  (0 = CNN output embeddings, 1..N = transformer layers)")
    ax.set_ylabel("softmax weight")
    ax.set_title(f"Learned layer weights — mean over {len(fold_layer_weights)} fold(s)")
    ax.set_xticks(idx)
    ax.legend()
    plt.tight_layout()
    plt.savefig(WORK_DIR / "fig_xlsr_layer_weights.png", dpi=150)
    plt.show()

    peak = int(np.argmax(mean_lw))
    print(f"Highest-weighted hidden state: index {peak} "
          f"(weight {mean_lw[peak]:.3f} vs {1 / len(mean_lw):.3f} uniform)")
    top5 = np.argsort(mean_lw)[::-1][:5]
    print("Top 5: " + ", ".join(f"idx {i} = {mean_lw[i]:.3f}" for i in top5))
    if peak <= len(mean_lw) * 0.25:
        print("  Weight sits low in the stack — the signal is closer to acoustics than to the")
        print("  higher-level representations, which is common for paralinguistic tasks.")
    elif peak >= len(mean_lw) * 0.75:
        print("  Weight sits high in the stack, which is less usual for a paralinguistic task —")
        print("  worth a second look at whether the model is keying on content rather than voice.")
    else:
        print("  Weight sits mid-stack, which is the usual place for paralinguistic information")
        print("  in a wav2vec2-family encoder.")
    pd.DataFrame(lw_arr, columns=[f"state_{i}" for i in idx]).to_csv(
        WORK_DIR / "xlsr_layer_weights.csv", index=False)
else:
    print(f"POOL_MODE is {POOL_MODE!r}, so there are no learned layer weights to plot.")
    print("Set POOL_MODE = 'weighted_sum' in the config cell to get this analysis.")


## 6. Test-set evaluation

Per-class precision/recall/F1, plus the two numbers a binary screening task is actually judged on:
**sensitivity** (of the children who have ASD, how many were caught) and **specificity** (of the TD
children, how many were correctly cleared). ASD is the positive class.

The paper's 0.769 is printed as a reference point only — it is a 3-class result and this is a 2-class
task, so the two are not comparable in either direction.


In [ ]:
# all_preds / all_labels were pooled across folds in the training cell above, so with USE_CV=True
# this table covers EVERY participant (each tested exactly once) rather than a single ~18-child
# test split.
test_acc = accuracy_score(all_labels, all_preds)
scope = f"pooled over {len(FOLDS)} folds — all {len(set(seg_df['participant']))} participants" \
        if USE_CV else "single held-out split"
print(f"Overall test accuracy: {test_acc:.3f}  ({scope})")
print(f"  chance for this binary task is {1 / N_CLASSES:.1%}, so the margin over chance is "
      f"{test_acc - 1 / N_CLASSES:+.3f}")
print(f"  [reference only] the paper reports 0.769 on the 3-CLASS task (ASD/ADHD/TD) from a single "
      f"split. That is a different problem with a 33.3% chance line — not a baseline this number "
      f"beats or misses.")
if USE_CV:
    print(f"  per-fold mean +/- std: {fold_df['accuracy'].mean():.3f} "
          f"+/- {fold_df['accuracy'].std():.3f}")

table3 = pd.DataFrame({
    "Precision": precision_score(all_labels, all_preds, average=None, zero_division=0),
    "Recall": recall_score(all_labels, all_preds, average=None, zero_division=0),
    "F1 Score": f1_score(all_labels, all_preds, average=None, zero_division=0),
}, index=[id2label[i] for i in sorted(id2label)]).round(2)
table3.to_csv(WORK_DIR / "table3_deep_learning_results.csv")

# Binary screening metrics. Only meaningful because the task has exactly two classes — there is no
# single sensitivity/specificity pair in the paper's 3-class setting, which is why this block is
# specific to this notebook.
POSITIVE_CLASS = "ASD"          # the condition being screened FOR
NEGATIVE_CLASS = "TD"
pos, neg = label2id[POSITIVE_CLASS], label2id[NEGATIVE_CLASS]

cm_bin = confusion_matrix(all_labels, all_preds, labels=[neg, pos])
(tn, fp), (fn, tp) = cm_bin
sensitivity = tp / (tp + fn) if (tp + fn) else float("nan")   # recall of ASD
specificity = tn / (tn + fp) if (tn + fp) else float("nan")   # recall of TD
ppv = tp / (tp + fp) if (tp + fp) else float("nan")
npv = tn / (tn + fn) if (tn + fn) else float("nan")
balanced_acc = (sensitivity + specificity) / 2

print(f"\nBinary screening metrics (positive class = {POSITIVE_CLASS}, segment level):")
print(f"  sensitivity / recall({POSITIVE_CLASS}) : {sensitivity:.3f}   "
      f"({tp} of {tp + fn} {POSITIVE_CLASS} segments caught)")
print(f"  specificity / recall({NEGATIVE_CLASS})  : {specificity:.3f}   "
      f"({tn} of {tn + fp} {NEGATIVE_CLASS} segments correctly cleared)")
print(f"  PPV (precision, {POSITIVE_CLASS})       : {ppv:.3f}")
print(f"  NPV                          : {npv:.3f}")
print(f"  balanced accuracy            : {balanced_acc:.3f}   "
      f"(mean of the two recalls — unlike plain accuracy this is not flattered by class imbalance)")

binary_metrics = pd.Series({
    "sensitivity": sensitivity, "specificity": specificity, "PPV": ppv, "NPV": npv,
    "balanced_accuracy": balanced_acc, "accuracy": test_acc,
}).round(4)
binary_metrics.to_csv(WORK_DIR / "binary_screening_metrics.csv", header=["value"])

table3

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=[id2label[i] for i in sorted(id2label)],
            yticklabels=[id2label[i] for i in sorted(id2label)], ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"XLS-R 300m (full fine-tune) confusion matrix — binary {' vs '.join(CLASSES_SORTED)}\n"
             f"(test acc {test_acc:.3f}, chance {1 / N_CLASSES:.1%})")
plt.tight_layout()
plt.savefig(WORK_DIR / "fig_xlsr_confusion_matrix.png", dpi=150)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
table3.plot(kind="bar", ax=ax)
ax.set_title(f"Per-class deep learning performance — XLS-R, binary {' vs '.join(CLASSES_SORTED)}")
ax.set_ylabel("Score"); ax.set_ylim(0, 1)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(WORK_DIR / "fig_table3_reproduction.png", dpi=150)
plt.show()


## 7. Explainability — frequency-band occlusion (reproduces Figure 2a)

Zeroes out a spectral band via a bandstop filter, re-runs the classifier, and measures the drop/gain in
predicted probability for the true class relative to the unaltered signal.


In [ ]:
from scipy.signal import butter, sosfiltfilt

FREQ_BANDS = [(100, 300), (300, 600), (600, 1000), (1000, 2000), (2000, 4000), (4000, 8000)]

def band_stop(y, sr, low, high, order=4):
    high = min(high, sr / 2 - 1)
    sos = butter(order, [low, high], btype="bandstop", fs=sr, output="sos")
    # sosfiltfilt filters forward then backward and hands back a reversed VIEW with negative
    # strides; torch.tensor() rejects those outright. ascontiguousarray makes a normal copy.
    return np.ascontiguousarray(sosfiltfilt(sos, y))

@torch.no_grad()
def predict_proba(y):
    # eval() here, not just in the test cell above: dropout and SpecAugment are both active in
    # train mode, which makes two calls on identical audio return different probabilities. The
    # occlusion delta would then include that noise rather than the occlusion's effect.
    model.eval()
    # random_crop=False: this is called twice per measurement (clean vs occluded) and a random
    # window each time would make the delta reflect the window, not the occlusion.
    y_padded, valid_len = pad_or_crop(y, TARGET_SR, random_crop=False)
    # Same normalisation the model was trained under. Without this the occlusion analysis would
    # be probing XLS-R with a distribution it never saw -- and band-stopping changes the
    # waveform's variance, so normalising AFTER the filter (as here, via pad_or_crop's output)
    # is also what keeps the clean/occluded pair comparable on level rather than on loudness.
    y_padded = normalize_waveform(y_padded, valid_len)
    x = torch.tensor(y_padded, dtype=torch.float32).unsqueeze(0).to(DEVICE)
    mask = (torch.arange(x.shape[1]) < valid_len).long().unsqueeze(0).to(DEVICE)
    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=AMP_ENABLED):
        logits = model(x, attention_mask=mask)
    return torch.softmax(logits.float(), dim=-1).cpu().numpy()[0]

freq_occlusion_rows = []
sample_per_class = test_df.groupby("class").head(10)  # cap for runtime; raise for a fuller sweep
for _, row in sample_per_class.iterrows():
    y, sr = librosa.load(row["segment_path"], sr=TARGET_SR, mono=True)
    true_idx = label2id[row["class"]]
    base_proba = predict_proba(y)[true_idx]
    for low, high in FREQ_BANDS:
        y_occ = band_stop(y, sr, low, high)
        occ_proba = predict_proba(y_occ)[true_idx]
        freq_occlusion_rows.append({
            "class": row["class"], "band": f"{low}-{high}", "delta_prob": base_proba - occ_proba,
        })

freq_occ_df = pd.DataFrame(freq_occlusion_rows)
freq_occ_summary = freq_occ_df.groupby(["class", "band"])["delta_prob"].mean().unstack("band")
band_order = [f"{l}-{h}" for l, h in FREQ_BANDS]
freq_occ_summary = freq_occ_summary[band_order]
freq_occ_summary


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
freq_occ_summary.T.plot(kind="bar", ax=ax)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Figure 2a reproduction — frequency-band occlusion sensitivity")
ax.set_ylabel(r"$\Delta$ prob (true class)")
ax.set_xlabel("Frequency band (Hz)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(WORK_DIR / "fig2a_frequency_occlusion.png", dpi=150)
plt.show()


## 8. Explainability — time occlusion (reproduces Figure 2b)

In [ ]:
def time_occlusion_curve(y, sr, true_idx, window_ms=200, max_len_s=MAX_DURATION_S):
    y, valid_len = pad_or_crop(y, sr, max_len_s, random_crop=False)
    base_proba = predict_proba(y)[true_idx]
    win = int(window_ms / 1000 * sr)
    # Only sweep windows inside the REAL audio. Occluding pure zero-padding changes nothing, and
    # averaging those no-op points into the curve flattens it toward zero.
    n_windows = max(1, int(valid_len) // win)
    deltas, times = [], []
    for i in range(n_windows):
        y_occ = y.copy()
        y_occ[i * win:(i + 1) * win] = 0.0
        # predict_proba normalises internally, so the zeroed window shifts this clip's mean and
        # variance slightly -- exactly as it would for any real occlusion. Pre-normalising here
        # instead would make the clean and occluded passes use different statistics.
        occ_proba = predict_proba(y_occ)[true_idx]
        deltas.append(base_proba - occ_proba)
        times.append(i * window_ms / 1000)
    return times, deltas

time_occ_rows = []
for _, row in sample_per_class.iterrows():
    y, sr = librosa.load(row["segment_path"], sr=TARGET_SR, mono=True)
    true_idx = label2id[row["class"]]
    times, deltas = time_occlusion_curve(y, sr, true_idx)
    for t, d in zip(times, deltas):
        time_occ_rows.append({"class": row["class"], "time_s": t, "delta_prob": d})

time_occ_df = pd.DataFrame(time_occ_rows)
time_occ_curve = time_occ_df.groupby(["class", "time_s"])["delta_prob"].mean().reset_index()

fig, ax = plt.subplots(figsize=(9, 5))
for cls in CLASSES_SORTED:
    sub = time_occ_curve[time_occ_curve["class"] == cls]
    ax.plot(sub["time_s"], sub["delta_prob"], label=cls, marker="o", markersize=3)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title(r"Figure 2b reproduction — time occlusion ($\Delta$ prob over time)")
ax.set_xlabel("Time (s)"); ax.set_ylabel(r"$\Delta$ prob (true class)")
ax.legend()
plt.tight_layout()
plt.savefig(WORK_DIR / "fig2b_time_occlusion.png", dpi=150)
plt.show()


## 9. Summary

This run's numbers, with the paper's reported accuracy shown **as a labelled reference, not a
baseline**. There are now two independent reasons the two are not comparable:

1. The paper's 0.769 is a **3-class** result (chance 33.3%); this is **2-class** (chance 50.0%).
2. The paper used **WavLM-Base-Plus** with a light fine-tune; this is **XLS-R 300m fully unfrozen** —
   a different model, 3.3x the parameters, and a different training recipe.

A higher number here does not mean the reproduction beat the paper, and it does not by itself mean
XLS-R beats WavLM either. For that second claim you would need to run the WavLM notebook and this one
on the **same binary task, same folds, same seed**, and compare those two directly — which is a clean
experiment worth doing, and the one your thesis can actually defend.


In [ ]:
TASK_NAME = " vs ".join(CLASSES_SORTED)
CHANCE = 1 / N_CLASSES

summary_rows = [
    {"Pipeline": "WavLM-Base-Plus — paper, 3-class ASD/ADHD/TD [reference, NOT comparable]",
     "Accuracy": 0.769, "Macro F1": np.nan, "Chance": 1 / 3},
    {"Pipeline": f"XLS-R 300m (full fine-tune) — this run, binary {TASK_NAME} "
                 f"({'pooled ' + str(len(FOLDS)) + '-fold CV' if USE_CV else 'single split'})",
     "Accuracy": test_acc,
     "Macro F1": f1_score(all_labels, all_preds, average="macro", zero_division=0),
     "Chance": CHANCE},
]
if USE_CV:
    summary_rows.append({
        "Pipeline": f"XLS-R 300m (full fine-tune) — this run, binary {TASK_NAME} "
                    f"(mean of {len(FOLDS)} folds)",
        "Accuracy": fold_df["accuracy"].mean(),
        "Macro F1": fold_df["macro_f1"].mean(),
        "Chance": CHANCE,
    })
summary_df = pd.DataFrame(summary_rows).round(4)
# How far above its OWN chance line each row sits. This is the only column on which a 2-class and a
# 3-class result can be put side by side at all, and even then only loosely.
summary_df["Above chance"] = (summary_df["Accuracy"] - summary_df["Chance"]).round(4)
summary_df.to_csv(WORK_DIR / "final_comparison_vs_paper.csv", index=False)

print(f"This notebook's task: binary {TASK_NAME} (chance {CHANCE:.1%}).")
print(f"Encoder: {XLSR_CHECKPOINT}, fully fine-tuned "
      f"(CNN front-end frozen: {FREEZE_FEATURE_EXTRACTOR}), pooling: {POOL_MODE}.")
print("The paper's row is 3-class ASD/ADHD/TD with WavLM-Base-Plus and is listed for reference "
      "only — do not report this run as reproducing or exceeding it.")
if USE_CV:
    print(f"Fold spread — accuracy {fold_df['accuracy'].min():.3f} to "
          f"{fold_df['accuracy'].max():.3f} (std {fold_df['accuracy'].std():.3f}). The paper's "
          f"0.769 comes from one split and carries no comparable error bar.")
summary_df


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
plot_df = summary_df.dropna(subset=["Accuracy"])
sns.barplot(data=plot_df, y="Pipeline", x="Accuracy", ax=ax, palette="viridis")
ax.axvline(CHANCE, color="red", ls="--", lw=1,
           label=f"chance, binary task ({CHANCE:.0%})")
ax.axvline(1 / 3, color="grey", ls=":", lw=1, label="chance, paper's 3-class task (33%)")
ax.legend(loc="lower right", fontsize=8)
ax.set_title(f"This run (XLS-R, binary {TASK_NAME}) vs. the paper's 3-class WavLM accuracy\n"
             f"Different tasks, different encoders, different chance lines — "
             f"the paper bar is a reference, not a baseline",
             fontsize=10)
plt.tight_layout()
plt.savefig(WORK_DIR / "fig_final_comparison.png", dpi=150)
plt.show()


## Notes / next steps

### On the encoder swap
- **XLS-R 300m has seen Bangla**; WavLM-Base-Plus has not. That is the whole reason for this
  notebook. If your corpus is not Bangla (or not predominantly non-English), the swap is not
  automatically an upgrade — XLS-R is 3.3x the parameters and massively slower to fine-tune.
- **Nothing is frozen**, as requested: all 24 transformer layers, the CNN feature extractor and the
  head all receive gradient. `POOL_MODE = "weighted_sum"` is what makes that true in practice —
  with fixed-layer pooling, every layer above the pooling point gets zero gradient no matter what
  `requires_grad` says.
- **If training diverges** (`nan` loss, or val accuracy stuck at chance), set
  `FREEZE_FEATURE_EXTRACTOR = True` first. Every official wav2vec2/XLS-R recipe freezes the CNN
  front-end; it is the single most likely cause and it still leaves all 24 transformer layers
  trainable. Second and third choices: `LR = 5e-6`, `WARMUP_RATIO = 0.2`.
- **Check the layer-weight plot** (Section 5b). If the weight concentrates around some layer *k*,
  you can switch to `POOL_MODE = "fixed_layer"` with `XLSR_LAYER = k` and delete the layers above it
  — a much cheaper model at, usually, very little cost in accuracy. That is a legitimate result to
  report, not just an optimisation.

### On comparing numbers
- **Do not report this run's accuracy against the paper's 0.769.** Two different things changed at
  once: the task (3-class → 2-class) and the encoder (WavLM → XLS-R). Section 9 keeps them labelled
  and separate.
- **The comparison your thesis actually wants** is WavLM vs XLS-R on *this* binary task, same folds,
  same `RANDOM_SEED`. Both notebooks write `cv_fold_results.csv` — run both, then compare per-fold
  and report a paired test across the 5 folds rather than two bare means.
- **Sensitivity and specificity** (Section 6, ASD as the positive class) remain the more defensible
  headline numbers for a screening task than accuracy, especially with unequal group sizes.

### On practicalities
- **Checkpoints are ~1.2 GB each** (315 M params in fp32), one per fold. Five folds is ~6 GB in
  `/kaggle/working`, against Kaggle's ~20 GB limit. If you add folds or hit the limit, delete each
  fold's checkpoint after its test evaluation.
- **Runtime is the main cost.** Full fine-tuning a 315 M encoder with gradient checkpointing runs
  roughly 10-20x a WavLM head-only epoch. Early stopping (patience 3) usually cuts folds well short
  of the 10-epoch cap — watch the first fold's per-epoch time before committing to all five.
- **A checkpoint trained by this notebook has a 2-unit output layer and an XLS-R backbone.** It is
  not loadable by an inference notebook expecting WavLM — the architecture differs, not just the
  output width.
- All generated tables/figures are saved under `/kaggle/working/` for direct inclusion in the thesis.
